# BytePlus Registration v2 - Stealth
Run all cells. Uses Playwright with real Chrome fingerprint.
Google Colab IPs are clean and not rate-limited by BytePlus.

In [ ]:
# Install deps
!pip install -q playwright requests
!playwright install chromium
!playwright install-deps chromium

In [ ]:
import asyncio, re, time, random, string, json, requests
from playwright.async_api import async_playwright

ARK = 'https://ark.cn-beijing.volces.com/api/v3'
MODEL = 'doubao-seedance-2-5-260628'

# Check IP
ip_info = requests.get('https://ipinfo.io/json', timeout=10).json()
print(f'IP: {ip_info.get("ip")} | {ip_info.get("org")} | {ip_info.get("country")}')

def gen_rand(n):
    return ''.join(random.choices(string.ascii_lowercase + string.digits, k=n))

def test_key(k):
    r = requests.post(f'{ARK}/contents/generations/tasks',
        headers={'Authorization': f'Bearer {k}', 'Content-Type': 'application/json'},
        json={'model': MODEL, 'content': [{'type': 'text', 'text': 'cat'}]}, timeout=14)
    ec = ''
    try: ec = r.json().get('error', {}).get('code', '')
    except: pass
    return r.status_code, ec

async def register_account():
    email_addr = None
    otp_code = None
    
    # Get guerrillamail email
    sess = requests.Session()
    r = sess.get('https://api.guerrillamail.com/ajax.php',
                 params={'f': 'get_email_address'}, timeout=10)
    gdata = r.json()
    email_addr = gdata.get('email_addr', '')
    sid = gdata.get('sid_token', '')
    print(f'Email: {email_addr}')
    
    username = f'bpdev{gen_rand(8)}'
    password = f'BpP1us!2026#A{gen_rand(4)}'
    print(f'Username: {username}')
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage',
                  '--disable-blink-features=AutomationControlled']
        )
        ctx = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
            extra_http_headers={
                'sec-ch-ua': '"Google Chrome";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
                'sec-ch-ua-mobile': '?0',
                'sec-ch-ua-platform': '"Windows"',
                'Accept-Language': 'en-US,en;q=0.9',
            }
        )
        page = await ctx.new_page()
        
        # Intercept RegisterAccountV2 response
        reg_result = {}
        async def on_response(resp):
            if 'RegisterAccountV2' in resp.url or 'account_management' in resp.url:
                try:
                    body = await resp.text()
                    reg_result['body'] = body
                    reg_result['url'] = resp.url
                    print(f'REG RESP: {body[:200]}')
                except: pass
        page.on('response', on_response)
        
        print('Loading signup page...')
        await page.goto('https://console.byteplus.com/auth/signup', timeout=30000)
        await asyncio.sleep(2)
        
        # Fill email
        await page.fill('input[type="email"], input[placeholder*="email" i], input[name="email"]', email_addr)
        await asyncio.sleep(1)
        
        # Fill username
        try:
            await page.fill('input[placeholder*="username" i], input[name="username"]', username, timeout=3000)
        except: pass
        
        # Fill password
        try:
            await page.fill('input[type="password"]', password, timeout=3000)
        except: pass
        
        # Click Continue / Send OTP
        for btn_text in ['Continue', 'Get code', 'Send code', 'Sign up', 'Next', 'Verify email']:
            try:
                btn = await page.get_by_text(btn_text, exact=False).first.element_handle(timeout=2000)
                if btn:
                    await btn.click()
                    print(f'Clicked: {btn_text}')
                    break
            except: pass
        
        # Wait for OTP
        print('Waiting for OTP...')
        for i in range(20):
            await asyncio.sleep(3)
            # Check guerrillamail
            try:
                em_r = sess.get('https://api.guerrillamail.com/ajax.php',
                               params={'f': 'get_email_list', 'offset': 0, 'sid_token': sid}, timeout=8)
                emails = em_r.json().get('list', [])
                for em in emails:
                    if 'byteplus' in em.get('mail_from', '').lower() or 'byte' in em.get('mail_from', '').lower() or 'code' in em.get('mail_subject', '').lower():
                        # Fetch full email
                        mid = em.get('mail_id')
                        body_r = sess.get('https://api.guerrillamail.com/ajax.php',
                                         params={'f': 'fetch_email', 'email_id': mid, 'sid_token': sid}, timeout=8)
                        body = body_r.json().get('mail_body', '')
                        otp_match = re.search(r'\b(\d{6})\b', body)
                        if otp_match:
                            otp_code = otp_match.group(1)
                            print(f'OTP: {otp_code}')
                            break
            except Exception as e:
                print(f'  mail check err: {e}')
            if otp_code:
                break
            print(f'  checking {(i+1)*3}s...')
        
        if not otp_code:
            print('No OTP received!')
            await browser.close()
            return None
        
        # Fill OTP
        try:
            otp_input = await page.query_selector('input[placeholder*="code" i], input[maxlength="6"]')
            if otp_input:
                await otp_input.fill(otp_code)
                print(f'Filled OTP: {otp_code}')
            else:
                # Try individual digit inputs
                inputs = await page.query_selector_all('input[maxlength="1"]')
                for idx, digit in enumerate(otp_code[:len(inputs)]):
                    await inputs[idx].fill(digit)
                print(f'Filled OTP digits')
        except Exception as e:
            print(f'OTP fill err: {e}')
        
        # Click Verify / Register
        for btn_text in ['Verify', 'Register', 'Create account', 'Sign up', 'Submit']:
            try:
                btn = await page.get_by_text(btn_text, exact=False).first.element_handle(timeout=2000)
                if btn:
                    await btn.click()
                    print(f'Clicked: {btn_text}')
                    break
            except: pass
        
        await asyncio.sleep(5)
        url = page.url
        print(f'Final URL: {url}')
        print(f'Reg result: {reg_result}')
        
        # Check if registered
        success = 'login' in url or 'dashboard' in url or 'console' in url.lower()
        if 'Error' not in str(reg_result) and 'OperationTooFrequent' not in str(reg_result):
            if otp_code:
                success = True
        
        await browser.close()
        
        if success:
            return {'email': email_addr, 'username': username, 'password': password, 'otp': otp_code, 'url': url}
        return None

result = await register_account()
print(f'Result: {result}')

In [ ]:
# After registration - check for ARK API key
# (run this if registration succeeded)
if result:
    print(f'Registered: {result["email"]}')
    print('Now need to:')
    print('1. Login to BytePlus ARK console')
    print('2. Create API key')
    print('3. Enable Seedance 2.5 model')
    print('Go to: https://console.byteplus.com/ark/region:ark+ap-southeast-1/openManagement')
else:
    print('Registration failed')